# Quantum-Hybrid CVRP Solver — qCourier Yale Hackathon 2026

**Hybrid structure:**
- **Classical:** encode locations → build distance matrix → compute minimum vehicles → sweep clustering → assign vehicles
- **Quantum (QAOA):** optimize the route ordering within each group (replaces the brute-force permutation search)

The route ordering problem within each group is a small TSP, which we formulate as a QUBO and solve with QAOA running on qBraid.

## Install dependencies

In [ ]:
# Uncomment and run once if packages are not installed
# !pip install qbraid qiskit qiskit-aer scipy numpy

## Imports

In [ ]:
import os
import time
import numpy as np
from itertools import permutations
from scipy.optimize import minimize

# Qiskit — circuit construction
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector

# qBraid — IonQ hardware access
from qbraid.runtime import IonQProvider

# Classical solver — all non-quantum logic lives here
from classical_solver import (
    encode_locations, build_distance_matrix, compute_num_groups,
    cluster_houses, assign_vehicles, total_distance
)

## Device Setup — IonQ Forte-1 via qBraid

IonQ Forte-1 is a trapped-ion QPU accessed through qBraid.  
Set your IonQ API key as an environment variable or pass it directly:
```
export IONQ_API_KEY="your_key_here"
```

In [ ]:
provider = IonQProvider()
device = provider.get_device("qpu.forte-1")
print("Connected to:", device)

---
## Classical Components

All classical logic (location encoding, distance matrix, sweep clustering, vehicle assignment) is imported directly from `classical_solver.py` — the same module used by `yale.ipynb`.

The **only** thing this notebook changes is `build_route`: instead of brute-force permutations, we solve the TSP within each group using **QAOA on IonQ Forte-1**.

---
## Quantum Components

### Step 1 — Build TSP QUBO

For a group of `k` customers, we define `k²` binary variables:
- `x[pos][c] = 1` if customer `c` is visited at position `pos`

**Constraints** (enforced via penalty term λ):
1. Each position has exactly one customer: `(Σ_c x[pos,c] - 1)² = 0`
2. Each customer appears exactly once: `(Σ_pos x[pos,c] - 1)² = 0`

**Objective:** minimize total route distance depot→c₁→...→cₖ→depot

In [ ]:
def build_tsp_qubo(group_customers, dist_matrix, depot_id=0, penalty=None):
    """
    Build upper-triangular QUBO matrix for TSP on a small group.
    penalty defaults to 2x the sum of all relevant distances.
    """
    k = len(group_customers)
    n = k * k
    Q = np.zeros((n, n))

    if penalty is None:
        all_dists = (
            [dist_matrix[(0, c)] for c in group_customers] +
            [dist_matrix[(c, 0)] for c in group_customers] +
            [dist_matrix[(c1, c2)] for c1 in group_customers for c2 in group_customers if c1 != c2]
        )
        penalty = 2 * sum(all_dists)

    def var(pos, c):
        return pos * k + c

    # Constraint 1: each position has exactly one customer
    for pos in range(k):
        for c in range(k):
            Q[var(pos, c), var(pos, c)] -= penalty
        for c1 in range(k):
            for c2 in range(c1 + 1, k):
                Q[var(pos, c1), var(pos, c2)] += 2 * penalty

    # Constraint 2: each customer visited exactly once
    for c in range(k):
        for pos in range(k):
            Q[var(pos, c), var(pos, c)] -= penalty
        for p1 in range(k):
            for p2 in range(p1 + 1, k):
                Q[var(p1, c), var(p2, c)] += 2 * penalty

    # Objective: depot → first customer
    for c, cust in enumerate(group_customers):
        Q[var(0, c), var(0, c)] += dist_matrix[(depot_id, cust)]

    # Objective: customer → customer (consecutive positions)
    for pos in range(k - 1):
        for c1, cust1 in enumerate(group_customers):
            for c2, cust2 in enumerate(group_customers):
                d = dist_matrix[(cust1, cust2)]
                q1, q2 = var(pos, c1), var(pos + 1, c2)
                if q1 <= q2:
                    Q[q1, q2] += d
                else:
                    Q[q2, q1] += d

    # Objective: last customer → depot
    for c, cust in enumerate(group_customers):
        Q[var(k - 1, c), var(k - 1, c)] += dist_matrix[(cust, depot_id)]

    return Q

### Step 2 — Convert QUBO to Ising Hamiltonian

Substituting `x_i = (1 - z_i) / 2` (where `z_i ∈ {-1, +1}`) transforms the QUBO into an Ising model that QAOA can directly optimize:

$$H_C = \sum_i h_i Z_i + \sum_{i<j} J_{ij} Z_i Z_j$$

In [ ]:
def qubo_to_ising(Q):
    """
    Convert upper-triangular QUBO matrix to Ising h, J, offset.
    x_i = (1 - z_i) / 2  =>  C_QUBO = C_Ising + offset
    """
    n = Q.shape[0]
    h = np.zeros(n)
    J = {}
    offset = 0.0

    for i in range(n):
        h[i] -= Q[i, i] / 2
        offset += Q[i, i] / 2
        for j in range(i + 1, n):
            if abs(Q[i, j]) > 1e-10:
                J[(i, j)] = Q[i, j] / 4
                h[i] -= Q[i, j] / 4
                h[j] -= Q[i, j] / 4
                offset += Q[i, j] / 4

    return h, J, offset

### Step 3 — Build QAOA Circuit

QAOA alternates between:
- **Cost unitary** `e^{-iγH_C}`: encodes the problem (ZZ rotations)
- **Mixer unitary** `e^{-iβH_M}`: explores the solution space (X rotations)

Starting from uniform superposition `|+⟩^⊗n`, `p` layers are applied.

In [ ]:
def build_qaoa_circuit(n_qubits, h, J, p=1):
    """
    Build p-layer QAOA circuit with parametrized gamma (cost) and beta (mixer) angles.
    Returns: (circuit, gamma_params, beta_params)
    """
    gammas = ParameterVector('γ', p)
    betas  = ParameterVector('β', p)

    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))  # uniform superposition

    for layer in range(p):
        # --- Cost unitary ---
        # Single-qubit Z rotations from h terms
        for i in range(n_qubits):
            if abs(h[i]) > 1e-10:
                qc.rz(2 * gammas[layer] * h[i], i)

        # Two-qubit ZZ rotations from J terms
        for (i, j), Jij in J.items():
            if abs(Jij) > 1e-10:
                qc.cx(i, j)
                qc.rz(2 * gammas[layer] * Jij, j)
                qc.cx(i, j)

        # --- Mixer unitary ---
        for i in range(n_qubits):
            qc.rx(2 * betas[layer], i)

    qc.measure_all()
    return qc, list(gammas), list(betas)

### Step 4 — Classical Optimization Loop

COBYLA minimizes the expectation value of `H_C` by tuning the QAOA angles. Each function evaluation runs one circuit on qBraid.

In [ ]:
def optimize_qaoa(qc, gammas, betas, h, J, device, p=1, shots=512, maxiter=50):
    """
    Optimize QAOA gamma/beta parameters using COBYLA.
    Each iteration submits one circuit to the qBraid device.
    """
    all_params = list(gammas) + list(betas)
    n = qc.num_qubits

    def expectation(params):
        param_dict = {all_params[i]: params[i] for i in range(len(params))}
        bound_qc = qc.assign_parameters(param_dict)
        job = device.run(bound_qc, shots=shots)
        result = job.result()
        counts = result.data.get_counts()

        energy, total = 0.0, sum(counts.values())
        for bitstring, count in counts.items():
            z = [1 - 2 * int(b) for b in reversed(bitstring)]
            e  = sum(h[i] * z[i] for i in range(n))
            e += sum(Jij * z[i] * z[j] for (i, j), Jij in J.items())
            energy += count * e
        return energy / total

    # Multiple restarts to avoid local minima
    best_params, best_val = None, float('inf')
    for _ in range(3):
        x0 = np.random.uniform(0, 2 * np.pi, 2 * p)
        res = minimize(expectation, x0, method='COBYLA', options={'maxiter': maxiter})
        if res.fun < best_val:
            best_val = res.fun
            best_params = res.x

    return best_params

### Step 5 — Decode QAOA Measurement into a Route

In [ ]:
def decode_route(qc, optimal_params, gammas, betas, group_customers, dist_matrix,
                 device, shots=1024):
    """
    Run the optimized QAOA circuit on qBraid, sample bitstrings,
    and decode the most-frequent valid permutation into a route.
    Falls back to classical brute-force if no valid bitstring is found.
    """
    all_params = list(gammas) + list(betas)
    param_dict = {all_params[i]: optimal_params[i] for i in range(len(optimal_params))}
    bound_qc = qc.assign_parameters(param_dict)

    job = device.run(bound_qc, shots=shots)
    result = job.result()
    counts = result.data.get_counts()

    k = len(group_customers)
    best_route, best_dist = None, float('inf')

    # Try bitstrings from most to least frequent
    for bitstring in sorted(counts, key=lambda b: -counts[b]):
        bits = [int(b) for b in reversed(bitstring)]
        x = np.array(bits).reshape(k, k)

        # Valid only if each row and column sums to 1 (permutation matrix)
        if not (np.all(x.sum(axis=1) == 1) and np.all(x.sum(axis=0) == 1)):
            continue

        route = [0] + [group_customers[int(np.argmax(x[pos]))] for pos in range(k)] + [0]
        d = sum(dist_matrix[(route[i], route[i + 1])] for i in range(len(route) - 1))
        if d < best_dist:
            best_dist, best_route = d, route

    # Classical fallback
    if best_route is None:
        print("  [fallback] No valid QAOA bitstring — using classical exact solver.")
        for perm in permutations(group_customers):
            route = [0] + list(perm) + [0]
            d = sum(dist_matrix[(route[i], route[i + 1])] for i in range(len(route) - 1))
            if d < best_dist:
                best_dist, best_route = d, route

    return best_route, result

---
## Full Quantum-Hybrid Solver

In [ ]:
def solve_cvrp_quantum(depot, customers, N, C, device, p=1, shots=512, maxiter=50, verbose=True):
    """
    Full quantum-hybrid CVRP solver.
    Classical: clustering.  Quantum (QAOA): route optimization per group.
    """
    locations  = encode_locations(depot, customers)
    dist_matrix = build_distance_matrix(locations)
    H = len(customers)
    G = compute_num_groups(H, C, N)

    if verbose:
        print(f"  G = min(ceil({H}/{C}), {N}) = {G} vehicle(s)")

    groups     = cluster_houses(locations, dist_matrix, G, C)
    assignment = assign_vehicles(groups, N)

    if verbose:
        print(f"  Groups: {groups}")

    routes = []
    resource_log = []  # track qubits, gates, time per group

    for v in sorted(assignment):
        for group in assignment[v]:
            k = len(group)

            if k == 1:
                # Trivial: only one customer, no ordering needed
                routes.append((v, [0, group[0], 0]))
                resource_log.append({"qubits": 0, "gates": 0, "time_s": 0,
                                     "note": "trivial (k=1)"})
                continue

            if verbose:
                print(f"\n  [QAOA] Vehicle {v}, group {group} (k={k}, n_qubits={k*k})")

            # Build QUBO and convert to Ising
            Q       = build_tsp_qubo(group, dist_matrix)
            h, J, _ = qubo_to_ising(Q)
            n_qubits = k * k

            # Build QAOA circuit
            qc, gammas, betas = build_qaoa_circuit(n_qubits, h, J, p=p)
            gate_count = qc.decompose().count_ops()

            if verbose:
                print(f"    Circuit: {n_qubits} qubits, depth={qc.depth()}, "
                      f"gates={sum(gate_count.values())} (before binding)")

            # Optimize QAOA parameters on qBraid
            t0 = time.time()
            optimal_params = optimize_qaoa(qc, gammas, betas, h, J,
                                           device, p=p, shots=shots, maxiter=maxiter)

            # Decode result
            route, result = decode_route(qc, optimal_params, gammas, betas,
                                         group, dist_matrix, device, shots=1024)
            elapsed = time.time() - t0

            routes.append((v, route))
            resource_log.append({
                "qubits": n_qubits,
                "gates":  sum(gate_count.values()),
                "time_s": round(elapsed, 2)
            })

            if verbose:
                d = sum(dist_matrix[(route[i], route[i+1])] for i in range(len(route)-1))
                print(f"    Route: {' -> '.join(str(n) for n in route)}  (dist={d:.4f})")

    dist = total_distance(routes, dist_matrix)

    if verbose:
        print("\n  Final routes:")
        for i, (v, route) in enumerate(routes):
            print(f"    r{i+1} (vehicle {v}): {' -> '.join(str(n) for n in route)}")
        print(f"  Total distance: {dist:.4f}")

    return routes, dist, resource_log

---
## Run All Instances

In [ ]:
depot = (0, 0)

instances = {
    1: {"customers": [(-2,2),(-5,8),(2,3)],                                                                               "N": 2, "C": 5},
    2: {"customers": [(-2,2),(-5,8),(2,3)],                                                                               "N": 2, "C": 2},
    3: {"customers": [(-2,2),(-5,8),(2,3),(5,7),(2,4),(2,-3)],                                                            "N": 3, "C": 2},
    4: {"customers": [(-2,2),(-5,8),(6,3),(4,4),(3,2),(0,2),(-2,3),(-4,3),(2,3),(2,7),(-2,5),(-1,4)], "N": 4, "C": 3},
}

all_results = {}
resource_table = []

for inst_num, params in instances.items():
    print(f"{'='*55}")
    print(f"Instance {inst_num}  |  N={params['N']}, C={params['C']}")
    print(f"{'='*55}")

    routes, dist, rlog = solve_cvrp_quantum(
        depot, params["customers"], params["N"], params["C"],
        device=device, p=1, shots=512, maxiter=50
    )

    all_results[inst_num] = {"routes": routes, "total_distance": dist}

    total_qubits = max((r["qubits"] for r in rlog), default=0)
    total_gates  = sum(r["gates"] for r in rlog)
    total_time   = sum(r["time_s"] for r in rlog)
    resource_table.append({
        "instance": inst_num,
        "qubits":   total_qubits,
        "gates":    total_gates,
        "time_s":   round(total_time, 2)
    })
    print()

## Resource Usage Table

In [ ]:
print(f"{'Instance':<12} {'# Qubits':<12} {'# Gates':<12} {'Time (s)':<12}")
print("-" * 48)
for row in resource_table:
    print(f"{row['instance']:<12} {row['qubits']:<12} {row['gates']:<12} {row['time_s']:<12}")

## Write Solution Files

In [ ]:
os.makedirs("solutions_quantum", exist_ok=True)

for inst_num, res in all_results.items():
    path = f"solutions_quantum/Instance{inst_num}.txt"
    with open(path, "w") as f:
        for i, (_, route) in enumerate(res["routes"], start=1):
            f.write("r" + str(i) + ": " + ", ".join(str(n) for n in route) + "\n")
    print(f"Written {path}")